## CODE U-NET FROM SCRATCH
```
Why U-Net?
Preserves spatial details through skip connections
Works well with limited data
Industry standard for medical and scientific segmentation

### 1. Imports

In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision import datasets
import matplotlib.pyplot as plt

```
Import	                Purpose
torch	                Core PyTorch - tensors, GPU support
torch.nn	        Neural network building blocks (Conv2d, ReLU, etc.)
transforms	        Image preprocessing (resize, convert to tensor)
DataLoader	        Batches data for training
datasets	        Access to OxfordIIITPet dataset
matplotlib.pyplot	Visualization
```

### 2. Dataset (IMPORTANT)

In [2]:
from torchvision.datasets import OxfordIIITPet

```
DATASET EXPLANATION
What is OxfordIIITPet Dataset?
This dataset contains images of cats and dogs with segmentation masks!

Dataset Structure:
OxfordIIITPet Dataset
├── images/
│   ├── Abyssinian_1.jpg
│   ├── Abyssinian_2.jpg
│   ├── ...
│   └── Yorkshire_Terrier_100.jpg
│
└── annotations/
    ├── trimaps/
    │   ├── Abyssinian_1.png
    │   ├── Abyssinian_2.png
    │   ├── ...
    │   └── Yorkshire_Terrier_100.png
    └── ...
What's in a Mask Image?
Mask Image (trimap) pixel values: 
- 1 = Pet (foreground)
- 2 = Background
- 3 = Border (not used in our case)

So each pixel tells us exactly what it belongs to!
Why This Dataset?
Your Happy/Sad Dataset	     OxfordIIITPet
❌ No masks	             ✅ Has masks
❌ Can't do segmentation    ✅ Perfect for segmentation
❌ Only classification	     ✅ Pixel-level labels
```

### 2.1 Load dataset:

In [3]:
transform = transforms.Compose([              # Define a series of transformations to apply to the images in the dataset
    transforms.Resize((128,128)),             # Resize images to a fixed size (128x128 in this case) to ensure uniform input size for the model
    transforms.ToTensor()                     # Convert PIL Image to tensor and normalize pixel values to [0, 1]
])

target_transform = transforms.Compose([       # For segmentation masks, we typically want to convert them to tensors without normalization, as they represent class labels rather than pixel intensities
    transforms.Resize((128,128)),
    transforms.PILToTensor()                  # Convert PIL Image to tensor without normalization because segmentation masks are typically in the form of integer labels
])

dataset = OxfordIIITPet(                      # Create an instance of the OxfordIIITPet dataset      
    root='./data',                            # Specify the root directory where the dataset will be downloaded
    split='trainval',                         # Use the 'trainval' split to include both training and validation data in other words, we are using the entire dataset for training and for validation we can later split it ourselves if needed
    target_types='segmentation',              # Specify that we want to load segmentation masks as the target labels for the dataset
    download=True, 
    transform=transform,                      # Apply the defined transformations to the images
    target_transform=target_transform         # Apply the defined transformations to the segmentation masks (target labels)
)

loader = DataLoader(dataset, batch_size=8, shuffle=True)       # Create a DataLoader to load the dataset in batches of 8 and shuffle the data for training

# DataLoader setup:
# batch_size=8: Process 8 image-mask pairs at once
# shuffle=True: Randomize order each epoch
# What one batch contains:
# images shape: (8, 3, 128, 128)  # 8 RGB images
# masks shape:  (8, 1, 128, 128)  # 8 masks (1 channel)

### 3. U-Net Building Blocks

#### Double Conv
This is the building block of the U-Net model. It consists of two convolutional layers with ReLU activation in between and it will be used multiple times in the U-Net model.

This double convolution block will maintain the output shape same as the input shape and channel size will be output channel size.


In [4]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),                 # First convolutional layer: takes in_ch input channels and produces out_ch output channels, with a kernel size of 3 and padding of 1 to maintain spatial dimensions of the input same as output
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU()                                               # output shape remains the same as input shape due to padding, and the number of channels is out_ch after the first convolution and remains out_ch after the second convolution
        )
    def forward(self, x):
        return self.conv(x)

```
What DoubleConv does:

Input: (C_in, H, W)
    ↓
Conv2d (3×3, padding=1): (C_out, H, W)  ← preserves size
    ↓
ReLU: (C_out, H, W)  ← adds non-linearity
    ↓
Conv2d (3×3, padding=1): (C_out, H, W)  ← second conv
    ↓
ReLU: (C_out, H, W)
    ↓
Output: (C_out, H, W)  ← same size, more features
Why two convolutions?

First conv: Detect basic patterns
Second conv: Refine and combine patterns
Two layers = more expressive power
Padding=1: Keeps spatial size same (H,W unchanged) 

#### U-Net Model

In [5]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder Path (Contracting or Downsampling Path)

        self.down1 = DoubleConv(3, 64)            # First layer of the encoder takes 3 input channels (RGB image) and produces 64 output channels (64, 128, 128): 64 feature maps of size 128x128
        self.pool1 = nn.MaxPool2d(2)              # Max pooling layer with a kernel size of 2, which reduces the spatial dimensions by half (128x128 to 64x64) while keeping the number of channels the same (64)
        
        self.down2 = DoubleConv(64, 128)          # Output shape: (128, 64, 64)
        self.pool2 = nn.MaxPool2d(2)              # Output shape: (128, 32, 32)
        
        # Bottleneck
        self.middle = DoubleConv(128, 256)        # Output shape: (256, 32, 32) : The bottleneck layer takes the output from the last encoder layer and processes it to capture the most abstract features of the input image, with 256 feature maps of size 32x32
        
        # Decoder Path (Expanding or Upsampling Path)
        
        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)       # Output shape: (128, 64, 64), Transposed convolutional layer that upsamples the feature maps from the bottleneck, reducing the number of channels from 256 to 128 and doubling the spatial dimensions from 32x32 to 64x64, 2 is the kernel size and stride=2 means it will upsample by a factor of 2
        self.conv1 = DoubleConv(256, 128)                          # Output shape: (128, 64, 64), DoubleConv layer with 256 input channels and 128 output channels , the input channels are 256 because we will concatenate the output of the upsampling layer (128 channels) with the corresponding feature maps from the encoder path (128 channels) which gives us a total of 256 input channels for this DoubleConv layer (skip connection)
        
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)        # Output shape (64, 128, 128), upsample size by 2
        self.conv2 = DoubleConv(128, 64)                           # Output shape (64, 128, 128), similar skip connection

        self.final = nn.Conv2d(64, 3, 1)                           # It is just a normal convolutional layer with 64 input channels and 3 output channels, Output shape (3, 128, 128), Final convolutional layer that takes the output from the last decoder layer (64 channels) and produces the final output with 3 channels (for RGB segmentation mask), with a kernel size of 1 to maintain the spatial dimensions of the input and output the same (128x128)

    def forward(self, x):
        d1 = self.down1(x)                            # output shape: (64, 128, 128)
        p1 = self.pool1(d1)                           # output shape: (64, 64, 64)
        
        d2 = self.down2(p1)                           # output shape: (128, 64, 64)
        p2 = self.pool2(d2)                           # output shape: (128, 32, 32) 
        
        mid = self.middle(p2)                         # output shape: (256, 32, 32)
        
        u1 = self.up1(mid)                            # output shape: (128, 64, 64)
        u1 = torch.cat([u1, d2], dim=1)               # output shape: (256, 64, 64) , Concatenate the upsampled feature maps (128 channels) with the corresponding feature maps from the encoder path (128 channels) along the channel dimension (dim=1) dim =1 because we are concatenating along the channel dimension , resulting in a total of 256 channels for the input to the next DoubleConv layer
        u1 = self.conv1(u1)                           # output shape: (128, 64, 64) , Process the concatenated feature maps through the DoubleConv layer to refine the features and reduce the number of channels back to 128
        
        u2 = self.up2(u1)                             # output shape: (64, 128, 128) , Upsample the feature maps again to increase the spatial dimensions back to 128x128 and reduce the number of channels to 64
        u2 = torch.cat([u2, d1], dim=1)               # output shape: (128, 128, 128) , skip connection with the first layer of encoder path
        u2 = self.conv2(u2)                           # output shape: (64, 128, 128) 

        return self.final(u2)                         # output shape: (3, 128, 128) , Pass the output through the final convolutional layer to produce the final segmentation mask with 3 channels (for RGB) and the same spatial dimensions as the input image (128x128)

### 4. Training Setup

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'   # Set the device to GPU if available, otherwise use CPU

model = UNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 5. Training Loop

In [8]:
for epoch in range(2):
    for images, masks in loader:
        images = images.to(device)
        
        masks = masks.squeeze(1)
        masks = masks - 1                            # Subtract 1 from the labels to make them zero-based (assuming the original labels are 1, 2, 3 for the three classes, we want them to be 0, 1, 2 for CrossEntropyLoss)
        masks = masks.long().to(device)               

        outputs = model(images)
        loss = criterion(outputs, masks)                  # copare the model's output with the ground truth masks to compute the loss, CrossEntropyLoss is used for multi-class segmentation tasks where the target masks contain class labels for each pixel
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 0.6595397591590881
Epoch 2, Loss: 0.48811638355255127


### 6. VISUALIZATION

In [ ]:
images, masks = next(iter(loader))
images = images.to(device)

with torch.no_grad():
    preds = model(images).argmax(1).cpu()

plt.figure(figsize=(10,5))

for i in range(3):
    plt.subplot(3,3,i*3+1)
    plt.imshow(images[i].cpu().permute(1,2,0))
    plt.title("Image")

    plt.subplot(3,3,i*3+2)
    plt.imshow(masks[i].squeeze(), cmap='gray')
    plt.title("Mask")

    plt.subplot(3,3,i*3+3)
    plt.imshow(preds[i], cmap='gray')
    plt.title("Prediction")

plt.show()

### 7. (OPTIONAL) Overlay

In [ ]:
plt.imshow(images[0].cpu().permute(1,2,0))
plt.imshow(preds[0], alpha=0.5, cmap='jet')
plt.title("Overlay")
plt.show()